Pobieranie danych CR ze stacji NMDB
https://www.nmdb.eu/nest/

created by:
Sławomir Stuglik

Last Big update:
2025-10-17

Dla OULU

In [ ]:
import os
from urllib import request

# Składnia URL do pobierania danych
# STARE
# oulu_body = [
#     "http://nest.nmdb.eu/draw_graph.php?formchk=1&stations[]=OULU&tabchoice=revori&dtype=corr_for_efficiency&tresolution=",
#     "&force=1&yunits=",
#     "&date_choice=bydate&start_day=",
#     "&start_month=",
#     "&start_year=",
#     "&start_hour=0&start_min=0&end_day=",
#     "&end_month=",
#     "&end_year=",
#     "&end_hour=0&end_min=0&output=ascii"
# ]

# ZMIANY OD lipca 2025
oulu_body = [
    "https://www.nmdb.eu/nest/draw_graph.php?formchk=1&stations[]=OULU&tabchoice=revori&dtype=corr_for_efficiency&tresolution=",
    "&force=1&yunits=",
    "&date_choice=bydate&start_day=",
    "&start_month=",
    "&start_year=",
    "&start_hour=0&start_min=0&end_day=",
    "&end_month=",
    "&end_year=",
    "&end_hour=0&end_min=0&output=ascii"
]


units = 0  # Dla innych obliczeń, wartość jednostek ustawiona na 1
# units = 1  # Dla PDF

def oulu_pack(year_begin, year_end, tresolution):
    """
    Pobiera dane dla każdego roku, dzieląc plik na sześć części: co 2 miesiące.
    Następnie łączy te pliki w jeden roczny plik CSV.
    """
    for one_year in range(year_begin, year_end + 1):
        file_parts = []
        for start_month in range(1, 13, 2):  # Przedziały: 1, 3, 5, 7, 9, 11 (co 2 miesiące)
            end_month = start_month + 2
            end_year = one_year

            # Jeśli end_month przekracza grudzień, przejście na następny rok
            if end_month > 12:
                end_month -= 12
                end_year += 1

            # Ustawienie dnia zakończenia na 1. dzień miesiąca końcowego
            end_day = 1

            req = request.Request(
                oulu_body[0] + str(tresolution) + oulu_body[1] + str(units) +
                oulu_body[2] + "1" + oulu_body[3] + str(start_month) + oulu_body[4] + str(one_year) +
                oulu_body[5] + str(end_day) + oulu_body[6] + str(end_month) + oulu_body[7] + str(end_year) +
                oulu_body[8]
            )

            with request.urlopen(req) as response:
                the_page = response.read()
            half_label = f"M{start_month:02d}-M{(end_month % 12 or 12):02d}"
            part_file = download_save(the_page, one_year, start_month, half_label)
            file_parts.append(part_file)

        # Łączenie częściowych plików w jeden plik roczny
        merge_files(file_parts, one_year)

def download_save(the_page, year, start_month, label):
    """
    Zapisuje dane odpowiedzi do pliku CSV i zwraca nazwę tego pliku.
    Pomija ostatni wiersz danych.
    """
    page = str(the_page)
    c1 = f"{year}-{start_month:02d}-01 00:00:00;"
    new = page[page.find(c1):]  # Wycinamy dane od pierwszej znalezionej godziny 00:00
    c2 = '</code'
    End = new.find(c2)
    text = new[0:End]
    text1 = text.replace(";", "; ")

    # Zapis do pliku
    new_folder = "csv_data_oulu/"
    os.makedirs(new_folder, exist_ok=True)
    file_name = f"part_{year}_{label}.csv"

    with open(new_folder + file_name, "w+") as f:
        lines = text1.split('\\n')
        for line in lines[:-2]:  # Pomija ostatni pełny wiersz danych (bo pobiera na zakładkę)
            if line.strip():
                print(line, file=f)

    return new_folder + file_name

def merge_files(file_list, year):
    """
    Łączy wszystkie pliki CSV z listy w jeden plik roczny.
    """
    merged_file_name = f"csv_data_oulu/{year}_full.csv"
    with open(merged_file_name, "w") as outfile:
        for i, fname in enumerate(file_list):
            with open(fname) as infile:
                for line_num, line in enumerate(infile):
                    outfile.write(line)

    print(f"Plik roczny {merged_file_name} został utworzony.")

def main():
    # tresolution = 1  # 1 minuta - od 1996
    tresolution = 5  # 5 minut - od 1970
    # tresolution = 60  # 1h - od 1964-05
    year_begin, year_end = 1960, 2025
    oulu_pack(year_begin, year_end, tresolution)

if __name__ == '__main__':
    main()

Łączenie kilku rocznych np 2000,2001,..,2024,2025 w 1 oulu_data.csv

In [ ]:
import os
import pandas as pd

station = "oulu"
folder = "csv_data_" + station

# --- wybór rozdzielczości (w minutach) ---
# tresolution = 1      # 1 minuta – od 1996
tresolution = 5    # 5 minut – od 1970
# tresolution = 60   # 1 godzina – od 1964

# mapowanie minut -> etykieta do nazw plików
tresolution_labels = {
    1: "1min",
    5: "5min",
    60: "1H"
}

# mapowanie rozdzielczości -> minimalny rok danych
tresolution_rules = {
    1: 1996,
    5: 1970,
    60: 1964
}

min_year = tresolution_rules[tresolution]
tresolution_label = tresolution_labels[tresolution]

# wszystkie pliki *_full.csv
files = [f for f in os.listdir(folder) if f.endswith("_full.csv")]

# filtr: tylko pliki od min_year
files_filtered = [
    f for f in files
    if int(f.split("_")[0]) >= min_year
]

# sortowanie po roku
files_sorted = sorted(files_filtered, key=lambda x: int(x.split("_")[0]))

dfs = []

for file in files_sorted:
    path = os.path.join(folder, file)
    # plik full ma 2 kolumny bez headerów - warto sie upewniać
    df = pd.read_csv(path, sep=";", header=None, names=["datetime", "value"])
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

# zapis
final_filename = station +"_data.csv"  # domyslnie
final_filename = f"{station}_{tresolution_label}_data.csv"
merged.to_csv(final_filename, index=False)
print(f'Połączono pliki -> {final_filename}')




---

Skrypt powyżej pozwala pobierać dane z każdej stacji - wystarczy zmienić  w zapytaniu fragment

*?formchk=1&stations[]=OULU*

na inną, np

*?formchk=1&stations[]=MOSC*

oraz nazwe katalogu docelowego
np z csv_data_oulu na csv_data_mosc

przykład dla MOSC poniżej



---

**Pobieranie danych MOSC**


In [ ]:
# mozna deklarowac raz by nie edytowac w pobieraniu i potem w laczeniu
tresolution = 6*60  # 6H

In [ ]:
import os
from urllib import request

# Składnia URL do pobierania danych
oulu_body = [
    "https://www.nmdb.eu/nest/draw_graph.php?formchk=1&stations[]=MOSC&tabchoice=revori&dtype=corr_for_efficiency&tresolution=",
    "&force=1&yunits=",
    "&date_choice=bydate&start_day=",
    "&start_month=",
    "&start_year=",
    "&start_hour=0&start_min=0&end_day=",
    "&end_month=",
    "&end_year=",
    "&end_hour=0&end_min=0&output=ascii"
]

units = 0  # Dla innych obliczeń, wartość jednostek ustawiona na 1

def oulu_pack(year_begin, year_end, tresolution):
    """
    Pobiera dane dla każdego roku, dzieląc plik na sześć części: co 2 miesiące.
    Następnie łączy te pliki w jeden roczny plik CSV.
    """
    for one_year in range(year_begin, year_end + 1):
        file_parts = []
        for start_month in range(1, 13, 2):  # Przedziały: 1, 3, 5, 7, 9, 11 (co 2 miesiące)
            end_month = start_month + 2
            end_year = one_year

            if end_month > 12:
                end_month -= 12
                end_year += 1

            end_day = 1

            req = request.Request(
                oulu_body[0] + str(tresolution) + oulu_body[1] + str(units) +
                oulu_body[2] + "1" + oulu_body[3] + str(start_month) + oulu_body[4] + str(one_year) +
                oulu_body[5] + str(end_day) + oulu_body[6] + str(end_month) + oulu_body[7] + str(end_year) +
                oulu_body[8]
            )

            with request.urlopen(req) as response:
                the_page = response.read()
            half_label = f"M{start_month:02d}-M{(end_month % 12 or 12):02d}"
            part_file = download_save(the_page, one_year, start_month, half_label)
            file_parts.append(part_file)

        merge_files(file_parts, one_year)

def download_save(the_page, year, start_month, label):
    """
    Zapisuje dane odpowiedzi do pliku CSV i zwraca nazwę tego pliku.
    Pomija ostatni wiersz danych.
    """
    page = str(the_page)
    c1 = f"{year}-{start_month:02d}-01 00:00:00;"
    new = page[page.find(c1):]
    c2 = '</code'
    End = new.find(c2)
    text = new[0:End]
    text1 = text.replace(";", "; ")

    new_folder = "csv_data_mosc/"
    os.makedirs(new_folder, exist_ok=True)
    file_name = f"part_{year}_{label}.csv"

    with open(new_folder + file_name, "w+") as f:
        lines = text1.split('\\n')
        for line in lines[:-2]:
            if line.strip():
                print(line, file=f)

    return new_folder + file_name

def merge_files(file_list, year):
    """
    Łączy wszystkie pliki CSV z listy w jeden plik roczny.
    """
    merged_file_name = f"csv_data_mosc/{year}_full.csv"
    with open(merged_file_name, "w") as outfile:
        for i, fname in enumerate(file_list):
            with open(fname) as infile:
                for line_num, line in enumerate(infile):
                    outfile.write(line)

    print(f"Plik roczny {merged_file_name} został utworzony.")

def main():
    # tresolution = 1  # minuta
    # year_begin, year_end = 2018, 2021
    try:
      tresolution  # próbujemy użyć zmiennej
    except NameError: # jesli nie zostało zadeklarowane wczesniej
        tresolution = 6 * 60

    year_begin, year_end = 1960, 2025
    oulu_pack(year_begin, year_end, tresolution)

if __name__ == '__main__':
    main()

Łączenie w 1

In [ ]:
import os
import pandas as pd

station = "mosc"
folder = "csv_data_" + station

# wszystkie pliki kończące się na _full.csv
files = [f for f in os.listdir(folder) if f.endswith("_full.csv")]

# sortowanie po numerze na początku nazwy
files_sorted = sorted(files, key=lambda x: int(x.split("_")[0]))

dfs = []

for file in files_sorted:
    path = os.path.join(folder, file)
    # plik full ma 2 kolumny bez headerów - warto sie upewniać
    df = pd.read_csv(path, sep=";", header=None, names=["datetime", "value"])
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

# zapis
if tresolution == 6*60:
  final_filename = station +"_data.csv"  # domyslnie
elif tresolution == 6*60:
  final_filename = station +"_1H"+"_data.csv" # gdy chcemy pobierac w kilku gestoscosciach (6H jak w publikacji, 1H, 1min itd)
else:
  step = int(tresolution // 60)
  final_filename = station +f"_{step}H"+"_data.csv" # dla 1960 najmniejszy krok to 2H
merged.to_csv(final_filename, index=False)
print(f'Połączono pliki -> {final_filename}')


***TEST Z ALL***

chya działa poprawnie

PRZYKŁADOWY LINK z 2 dni

https://www.nmdb.eu/nest/draw_graph.php?formchk=1&allstations=1&output=ascii&tabchoice=ori&dtype=corr_for_efficiency&date_choice=bydate&start_year=2023&start_month=09&start_day=01&start_hour=00&start_min=00&end_year=2023&end_month=09&end_day=02&end_hour=23&end_min=59&tresolution=1&yunits=0


In [ ]:
import os
import datetime
from urllib import request

# --- Ustawienia bazowe ---
BASE_URL = (
    "https://www.nmdb.eu/nest/draw_graph.php?"
    "formchk=1&allstations=1&tabchoice=revori&dtype=corr_for_efficiency"
    "&output=ascii&tresolution={tres}&yunits={yunits}"
    "&date_choice=bydate&start_day={sd}&start_month={sm}&start_year={sy}"
    "&start_hour=0&start_min=0&end_day={ed}&end_month={em}&end_year={ey}"
    "&end_hour=23&end_min=59"
)

DATA_FOLDER = "csv_data_allstations"
os.makedirs(DATA_FOLDER, exist_ok=True)

units = 0  # 0 = counts, 1 = %


# --- Funkcje pomocnicze ---

def fetch_period(start_date, end_date, tresolution):
    """Pobiera dane z NMDB dla zadanego okresu i pokazuje diagnostykę."""
    url = BASE_URL.format(
        tres=tresolution,
        yunits=units,
        sd=start_date.day,
        sm=start_date.month,
        sy=start_date.year,
        ed=end_date.day,
        em=end_date.month,
        ey=end_date.year,
    )
    # link do zobaczenia jak wyglada 1 "paczka"
    # print(f"\n[REQUEST] {url}")
    try:
        with request.urlopen(url) as response:
            status = response.status
            body = response.read().decode("utf-8", errors="ignore")
    except Exception as e:
        print(f"Błąd pobierania {start_date}-{end_date}: {e}")
        return ""

    # print(f"HTTP status: {status}")
    # print(f"Długość odpowiedzi: {len(body)} znaków")
    # print(f"Podgląd początku odpowiedzi:\n{body[:300]}\n")

    return body


def parse_nmdb_ascii(data):
    """
    Czyści dane NMDB:
    - wycina HTML (zachowuje tylko zawartość <pre><code>...</code></pre>)
    - usuwa komentarze (# ...)
    - zostawia tylko linie z nagłówkiem i danymi
    """
    # print("[parse_nmdb_ascii] start")
    start_tag = "<pre><code>"
    end_tag = "</code>"

    start_idx = data.find(start_tag)
    end_idx = data.find(end_tag)

    if start_idx == -1 or end_idx == -1:
        print("Nie znaleziono tagów <pre><code> ... </code>")
        return None, []

    content = data[start_idx + len(start_tag):end_idx].strip()
    # print(f" Wycięto fragment danych ({len(content.splitlines())} linii w <code>)")

    lines = content.splitlines()

    header_line = None
    data_lines = []

    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        if not header_line and not line.startswith("20"):
            # zamiana na separator ; i usuwanie nadmiarowych ;;;
            header_line = line.replace(" ", ";").replace("\t", ";")
            while ";;" in header_line:
                header_line = header_line.replace(";;", ";")
            continue

        if line[:4].isdigit():
            # dane: usuń wielokrotne średniki i dodaj spacje po nich
            clean_line = line.replace(";", "; ")
            while ";;" in clean_line:
                clean_line = clean_line.replace(";;", ";")
            data_lines.append(clean_line.strip())

    # print(f"Nagłówek: {header_line[:60] if header_line else 'brak'}")
    # print(f"Liczba linii danych: {len(data_lines)}")

    return header_line, data_lines


def nmdb_allstations(year_begin, year_end, tresolution=1, step_days=2):
    """Pobiera dane dla wszystkich stacji w krokach co `step_days` dni i zapisuje jeden scalony plik."""
    # keidys byl limit 20tys wierszy, nie wiem jak to dziala dla all czy tez jest wiec pobieram co 2 dni
    delta = datetime.timedelta(days=step_days)

    for year in range(year_begin, year_end + 1):
        date = datetime.date(year, 1, 1)
        year_end_date = datetime.date(year, 12, 31)

        merged_lines = []
        header = None

        while date <= year_end_date:
            next_date = min(date + delta - datetime.timedelta(days=1), year_end_date)
            label = f"{date:%m%d}-{next_date:%m%d}"

            print(f"Pobieranie: {label}")
            data = fetch_period(date, next_date, tresolution)
            if not data:
                date = next_date + datetime.timedelta(days=1)
                continue

            h, lines = parse_nmdb_ascii(data)
            if h and not header:
                header = h
            merged_lines.extend(lines)

            date = next_date + datetime.timedelta(days=1)

        # zapis pojedynczego pliku rocznego
        if header and merged_lines:
            out_path = os.path.join(DATA_FOLDER, f"{year}_allstations.csv")
            with open(out_path, "w", encoding="utf-8") as f:
                f.write("timestamp;" + header + "\n")
                for line in merged_lines:
                    f.write(line + "\n")

            print(f"Zapisano pełny plik roku {year}: {out_path}")
        else:
            print(f"Brak danych dla roku {year}")


def main():
    nmdb_allstations(2023, 2024, tresolution=1, step_days=2)


if __name__ == "__main__":
    main()
